In [17]:
# Import standard libraries
from dataclasses import replace
import os
from pathlib import Path
import sys
import time

# Third-party libraries
import gymnasium as gym
from gymnasium.wrappers import RecordEpisodeStatistics
import numpy as np
import torch

In [18]:
# Add the folder containing our envs/ and rl/ packages to the path
sys.path.append("/workspace/software")

# Import the custom environment and PPO training module
from envs.balance_bot_env import BalanceBotEnv
from rl.ppo_trainer import PPOConfig, evaluate, train, export_tb_plots_as_csv

In [19]:
# Settings
MJCF_PATH = Path("/workspace/mechanical/FreeCAD/bala2-fire/bala2-fire-simplified.xml")
SEED = 42
NUM_ENVS = 4               # Number of parallel environments. Only the first will be rendered.
STEPS_PER_ENV = 500_000    # Number of simulation steps to perform per environment

In [10]:
# Configure PPO
ppo_config = PPOConfig(
    exp_name = "balance-bot-ppo",  # Name of the experiment
    env_id = "BalanceBot-v0",      # Name of the environment
    seed = SEED,                   # Constant seed for reproducibility
    num_envs = NUM_ENVS,           # Number of parallel environments
    actor_hidden_layers = 2,       # Number of hidden layers in the actor network
    actor_hidden_size = 16,        # Number of nodes in each hidden layer in the actor
    critic_hidden_layers = 2,      # Number of hidden layers in the critic network
    critic_hidden_size = 16,       # Number of nodes in each hidden layer in the critic
    total_timesteps = NUM_ENVS * STEPS_PER_ENV,  # Total simulation steps (all envs and iterations)
    num_steps = 2048,              # Number of steps per rollout per env (2048 * 0.002s = ~4 sec)
    num_minibatches = 32,          # Number of minibatches for each training epoch
    update_epochs = 10,            # Number of epochs to update actor and critic for each iteration
    anneal_lr = True,              # Enable annealing (lower learning rate as training goes on)
    learning_rate = 3e-4,          # Initial learning rate, reduced by annealing (if enabled)
    gamma = 0.99,                  # Discount factor (future rewards are discounted by this amount)
    gae_lambda = 0.95,             # GAE blending: 0 = pure TD error, 1 = pure Monte Carlo
    clip_coef = 0.2,               # Limits policy ratio to prevent large actor updates
    value_clip = 1.0,              # Absolute bounds on value prediction change per update (critic)
    ent_coef = 0.0,                # How much entropy factors into total loss calculation
    vf_coef = 0.5,                 # How much the value loss factors into total loss calculation
    max_grad_norm = 0.5,           # Limits how much actor/critic parameters can change during an update
    checkpoint_interval = 50,      # Save model every 50 iterations
    save_model = True,             # Save the final model
    timestep = 0.000,              # Match MJCF opt.timestep for real-time rendering (or 0 for fast)
)

In [11]:
def make_balance_bot_env(render, **kwargs):
    """Function to create an environment for our balance bot"""
    # Create the environment and set the render mode
    env = BalanceBotEnv(
        mjcf_path    = MJCF_PATH,
        render_mode  = "human" if render else None,
        **kwargs
    )

    # Wrap in RecordEpisodeStatistics so we can log episodic returns in the 'info' dict
    return gym.wrappers.RecordEpisodeStatistics(env)

def make_envs(num_envs, **kwargs):
    """Create a SyncVectorEnv with num_envs balance bot environments."""
    env_factories = []
    for i in range(num_envs):
        env_factories.append(
            lambda render=(i==0), kw=kwargs: make_balance_bot_env(render, **kw)
        )
    return gym.vector.SyncVectorEnv(env_factories)

In [12]:
# Phase 1: Balance only (don't worry about position or rotation)
envs = make_envs(
    NUM_ENVS,
    pitch_penalty_coef=0.5,
    action_penalty_coef=0.01,
    position_penalty_coef=0.0,
    yaw_penalty_coef=0.0
)

# Choo choo train
result = train(ppo_config, envs=envs)

Run name: BalanceBot-v0__balance-bot-ppo__42__1777948842
TensorBoard: http://localhost:6006/#scalars&regexFilter=balance-bot-ppo
Checkpoint saved to runs/BalanceBot-v0__balance-bot-ppo__42__1777948842/checkpoint_iter0050.cleanrl_model
New best model saved (mean_return=6116.28) to runs/BalanceBot-v0__balance-bot-ppo__42__1777948842/best_model.cleanrl_model
Checkpoint saved to runs/BalanceBot-v0__balance-bot-ppo__42__1777948842/checkpoint_iter0100.cleanrl_model
New best model saved (mean_return=6624.04) to runs/BalanceBot-v0__balance-bot-ppo__42__1777948842/best_model.cleanrl_model
Checkpoint saved to runs/BalanceBot-v0__balance-bot-ppo__42__1777948842/checkpoint_iter0150.cleanrl_model
New best model saved (mean_return=9988.64) to runs/BalanceBot-v0__balance-bot-ppo__42__1777948842/best_model.cleanrl_model
Checkpoint saved to runs/BalanceBot-v0__balance-bot-ppo__42__1777948842/checkpoint_iter0200.cleanrl_model
New best model saved (mean_return=9993.20) to runs/BalanceBot-v0__balance-bot-

In [13]:
# Inspect what was saved
print(f"Best model: {result.best_model_path}")
print(f"Final model: {result.final_model_path}")
print(f"Best mean return: {result.best_mean_return:.2f}")

# Load best model if available, otherwise use final
if result.best_model_path is not None:
    result.agent.load_state_dict(
        torch.load(result.best_model_path, weights_only=True)
    )
    print(f"Loaded best model (mean_return={result.best_mean_return:.2f})")

Best model: runs/BalanceBot-v0__balance-bot-ppo__42__1777948842/best_model.cleanrl_model
Final model: runs/BalanceBot-v0__balance-bot-ppo__42__1777948842/balance-bot-ppo_final.cleanrl_model
Best mean return: 9993.20
Loaded best model (mean_return=9993.20)


In [14]:
# Switch to real-time timestep
eval_config = replace(ppo_config, timestep=0.005)

# Evaluate
returns = evaluate(
    result.agent, 
    eval_episodes=3, 
    config=eval_config, 
    envs=envs)

# Print 
print(f"Mean return: {np.nanmean(returns):.2f}")

Mean return: 2046.67


In [15]:
# Get the run directory
run_path = result.checkpoint_dir

# Export TensorBoard plots as CSV files
export_tb_plots_as_csv(run_path)

Exported charts_metrics.csv (5 metrics, 1149 steps)
Exported losses_metrics.csv (7 metrics, 244 steps)


In [10]:
# Phase 2: Update the position and rotation coefficients in the existing environments
for env_stat_wrapper in envs.envs:
    env = env_stat_wrapper.env
    env.position_penalty_coef = 0.001
    env.yaw_penalty_coef = 0.1

# Choo choo train
result = train(ppo_config, envs=envs, agent=result.agent)

Run name: BalanceBot-v0__balance-bot-ppo__42__1777930816
TensorBoard: http://localhost:6006/#scalars&regexFilter=balance-bot-ppo
Checkpoint saved to runs/BalanceBot-v0__balance-bot-ppo__42__1777930816/checkpoint_iter0050.cleanrl_model
New best model saved (mean_return=9968.98) to runs/BalanceBot-v0__balance-bot-ppo__42__1777930816/best_model.cleanrl_model
Checkpoint saved to runs/BalanceBot-v0__balance-bot-ppo__42__1777930816/checkpoint_iter0100.cleanrl_model
Checkpoint saved to runs/BalanceBot-v0__balance-bot-ppo__42__1777930816/checkpoint_iter0150.cleanrl_model
Checkpoint saved to runs/BalanceBot-v0__balance-bot-ppo__42__1777930816/checkpoint_iter0200.cleanrl_model
New best model saved (mean_return=9985.69) to runs/BalanceBot-v0__balance-bot-ppo__42__1777930816/best_model.cleanrl_model
Final model saved to runs/BalanceBot-v0__balance-bot-ppo__42__1777930816/balance-bot-ppo_final.cleanrl_model
Best model mean return: 9985.69, saved to runs/BalanceBot-v0__balance-bot-ppo__42__177793081

In [11]:
# Inspect what was saved
print(f"Best model: {result.best_model_path}")
print(f"Final model: {result.final_model_path}")
print(f"Best mean return: {result.best_mean_return:.2f}")

# Load best model if available, otherwise use final
if result.best_model_path is not None:
    result.agent.load_state_dict(
        torch.load(result.best_model_path, weights_only=True)
    )
    print(f"Loaded best model (mean_return={result.best_mean_return:.2f})")

Best model: runs/BalanceBot-v0__balance-bot-ppo__42__1777930816/best_model.cleanrl_model
Final model: runs/BalanceBot-v0__balance-bot-ppo__42__1777930816/balance-bot-ppo_final.cleanrl_model
Best mean return: 9985.69
Loaded best model (mean_return=9985.69)


In [12]:
# Switch to real-time timestep
eval_config = replace(ppo_config, timestep=0.005)

# Evaluate
returns = evaluate(
    result.agent, 
    eval_episodes=3, 
    config=eval_config, 
    envs=envs)

# Print 
print(f"Mean return: {np.nanmean(returns):.2f}")

Mean return: 2046.43


In [13]:
# Get the run directory
run_path = result.checkpoint_dir

# Export TensorBoard plots as CSV files
export_tb_plots_as_csv(run_path)

Exported runs/BalanceBot-v0__balance-bot-ppo__42__1777930816/charts_metrics.csv (5 metrics, 481 steps)
Exported runs/BalanceBot-v0__balance-bot-ppo__42__1777930816/losses_metrics.csv (7 metrics, 244 steps)


In [16]:
# Close the environments
for idx, env in enumerate(envs.envs):
    print(f"Closing env {idx}")
    env.env.close()

Closing env 0
Closing env 1
Closing env 2
Closing env 3
